In [ ]:
import pyspark.sql.functions as F

In [0]:
 
# ── Unity Catalog location (must match the other scripts in this pipeline) ──
# See the README's "Key concepts" section for what catalog/schema mean.
CATALOG = "use1_prod_artemis_catalog_3718194974443840" #change
SCHEMA = "tier1_raw" #change
 
# ── Source tables produced by earlier tasks in this job ─────────────────────
flights_table = f"{CATALOG}.{SCHEMA}.drone_mission_table"          # from Job 1 (1_Flight_table)
ortho_table = f"{CATALOG}.{SCHEMA}.drone_ortho_table"               # from 2_5_Update_ortho_table
plot_clipped_table = f"{CATALOG}.{SCHEMA}.drone_location_clipped_table"  # from 2_Location_table_gen
 
raw_flights_df = spark.table(flights_table)
raw_ortho_df = spark.table(ortho_table)
raw_plots_df = spark.table(plot_clipped_table)
 
# These columns together uniquely identify a single flight, and are used to
# join the three tables above.
master_keys = ['site', 'trial', 'season', 'flight_date']
 
# ── Step 1: Which flights actually have an orthomosaic ready? ───────────────
# Only flights whose orthomosaic has been built (ortho_exists == True) are
# eligible to move on to the location-clipping step.
ready_orthos_df = raw_ortho_df.filter(F.col("ortho_exists") == True).select(*master_keys)
 
# Join back to the full flight list to get all flight details for the
# flights that are ready to be clipped.
ready_for_clipping_df = raw_flights_df.join(ready_orthos_df, on=master_keys, how='inner')
 
# ── Step 2: Which flights already have their location clipping done? ───────
# If the "plots_exist" column is present, only count flights where it's True
# as actually finished. If the column isn't there yet (e.g. first run before
# any clips exist), treat the whole table as the "finished" list as-is.
if "plots_exist" in raw_plots_df.columns:
    finish_df = raw_plots_df.filter(F.col("plots_exist") == True)
else:
    finish_df = raw_plots_df
 
# ── Step 3: Find the flights that are ready but NOT yet clipped ────────────
# A "left anti join" keeps only rows from ready_for_clipping_df that have NO
# match in finish_df — i.e., flights that are ready for clipping but are
# still missing their clipped output.
missing_df = ready_for_clipping_df.join(
    finish_df,
    on=['flight_metadata_path'],
    how='left_anti'
)
 
# ── Step 4: Report the current status ───────────────────────────────────────
total_ready = ready_for_clipping_df.count()
total_finish = finish_df.count()
total_missing = missing_df.count()
 
print("-" * 50)
print(f" LOCATION CLIPPING INVENTORY REPORT:")
print(f"Flights with Ortomosaics (Ready to clip): {total_ready}")
print(f"Flights with locations already clipped: {total_finish}")
print(f"Flights pending locations clipping: {total_missing}")
print("-" * 50)
 
if total_missing > 0:
    display(missing_df)
else:
    print(" Everything is up to date! There are no pending flights for plot clipping.")

In [0]:
# This is the key orchestration logic: it decides whether the pipeline
# should continue to the clipping step or stop here.
if total_missing > 0:
    # Extract the paths using list comprehension (Serverless compatible)
    path_list = [row[0] for row in missing_df.select('flight_metadata_path').collect()]
 
    # Pass the list of pending flights, plus a "proceed" flag, to the next
    # task via Databricks Job task values. The downstream condition task
    # (check_pending_location_clips) reads "proceed" to decide whether to
    # continue to pending_location_clips_gen.
    dbutils.jobs.taskValues.set(key="missing_clips", value=path_list)
    dbutils.jobs.taskValues.set(key="proceed", value="true")
 
    print(f" Green flag: {len(path_list)} pending flights sent to the Clipping Node.")
 
else:
    # No pending flights — set an empty list and tell the condition task to
    # stop the pipeline here instead of running the clipping step.
    dbutils.jobs.taskValues.set(key="missing_clips", value=[])
    dbutils.jobs.taskValues.set(key="proceed", value="false")
 
    print(" Red flag: There are no pending flights. Stopping pipeline here.")